In [1]:
def remove_pos_tags(sentence):
    # keep only the word before "_"
    return " ".join(token.split("_")[0] for token in sentence.split())

def extract_pos_tags(sentence):
    # keep only the tag after "_"
    return " ".join(token.split("_")[1] for token in sentence.split())

def main():
    output_sentences = "matching/english/sentences1.txt"
    output_tags = "matching/english/tags.txt"

    with open(output_sentences, "w", encoding="utf-8") as fout_s, \
         open(output_tags, "w", encoding="utf-8") as fout_t:

        for i in range(1, 21+1):
            input_file = f"syntax_twins/batch{i}.txt"
            with open(input_file, "r", encoding="utf-8") as fin:
                for line in fin:
                    parts = line.strip().split("\t")
                    if len(parts) != 2:
                        print(f"{i=} Skipping malformed line:", repr(line))
                        continue


                    second_sentence = parts[1]

                    # remove POS tags (keep words only)
                    cleaned_sentence = remove_pos_tags(second_sentence)

                    # extract POS tags only
                    pos_sequence = extract_pos_tags(second_sentence)

                    fout_s.write(cleaned_sentence + "\n")
                    fout_t.write(pos_sequence + "\n")

if __name__ == "__main__":
    main()

In [2]:
import numpy as np
syn_group_ids_path = '/home/acevedo/syn-sem/datasets/txt/syn/second/matching/english/group_ids.txt' 
syn_group_ids = np.loadtxt(syn_group_ids_path,dtype=int)
syntax_twins_path = '/home/acevedo/syn-sem/datasets/txt/syn/third/matching/english/sentences1.txt'
syntax_twins = []

with open(syntax_twins_path) as f:
    for line in f:
        syntax_twins.append(line.strip())

        

In [3]:
from collections import Counter

counts = Counter(syntax_twins)
duplicates = [s for s, c in counts.items() if c > 1]

print("Number of duplicated sentences:", len(duplicates))
for s in duplicates:
    print(counts[s], "×", repr(s))


Number of duplicated sentences: 0


In [4]:
from collections import defaultdict

# Map: sentence -> list of indices where it appears
sentence_to_indices = defaultdict(list)

for i, s in enumerate(syntax_twins):
    sentence_to_indices[s].append(i)

# Now print only those that appear more than once
for sentence, idxs in sentence_to_indices.items():
    if len(idxs) > 1:
        print(f"Sentence: {sentence!r}")
        print(f"Indices: {idxs}")
        print("-" * 60)


In [7]:
### shuffling syntax twins

import numpy as np
from collections import defaultdict

# Load data
syn_group_ids_path = '/home/acevedo/syn-sem/datasets/txt/syn/second/matching/english/group_ids.txt'
syn_group_ids = np.loadtxt(syn_group_ids_path, dtype=int)

syntax_twins_path = '/home/acevedo/syn-sem/datasets/txt/syn/third/matching/english/sentences1.txt'
with open(syntax_twins_path) as f:
    syntax_twins = [line.strip() for line in f]


# --- Internal Shuffle ---
group_to_indices = defaultdict(list)

# 1. Group indices by group_id
for idx, gid in enumerate(syn_group_ids):
    group_to_indices[gid].append(idx)

# 2. Prepare output list
shuffled_twins = syntax_twins.copy()

# Random generator (reproducible if you set seed)
rng = np.random.default_rng(seed=42)

# 3. Shuffle sentences *within each group*
for gid, idxs in group_to_indices.items():
    shuffled_positions = idxs.copy()
    rng.shuffle(shuffled_positions)

    for original_idx, new_idx in zip(idxs, shuffled_positions):
        shuffled_twins[original_idx] = syntax_twins[new_idx]

# Write shuffled sentences back to the file
with open(syntax_twins_path, "w") as f:
    for sentence in shuffled_twins:
        f.write(sentence + "\n")